In [5]:
import nbformat

path = "/Users/rajugaru/Desktop/FullStack-AI/TestDummy/ai-assistant/notebooks/non_instruction_finetuning.ipynb"  # <-- put full path here

nb = nbformat.read(path, as_version=4)

# ✅ Remove notebook-level widget metadata
nb.metadata.pop("widgets", None)

# ✅ Clean every cell completely
for cell in nb.cells:
    # Remove widget metadata inside cells
    cell.metadata.pop("widgets", None)
    
    # Remove outputs (important!)
    if "outputs" in cell:
        cell["outputs"] = []
    
    # Reset execution count
    if "execution_count" in cell:
        cell["execution_count"] = None

# ✅ Write clean notebook
nbformat.write(nb, path)

print("✅ Notebook FULLY cleaned!")

✅ Notebook FULLY cleaned!


In [14]:
from unsloth import FastLanguageModel

# Define the path where you saved your adapter
adapter_path = "../models/non_instruction_adapter"

# Load the model and tokenizer from the saved adapter path
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = adapter_path,
    max_seq_length = 512, # Use the same max_seq_length as during training
    dtype = None,         # Auto-detect
    load_in_4bit = True,  # QLoRA: 4-bit quantization
)

print(f"Model and tokenizer loaded from {adapter_path}")
model.print_trainable_parameters()

==((====))==  Unsloth 2026.6.9: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Unsloth: Will load ../models/non_instruction_adapter as a legacy tokenizer.


Model and tokenizer loaded from ../models/non_instruction_adapter
trainable params: 12,615,680 || all params: 1,112,664,064 || trainable%: 1.1338


# Stage 1: Non-Instruction Fine-Tuning — Healthcare Domain

This notebook covers the first stage of the Healthcare FAQ Assistant fine-tuning pipeline. Here we adapt the base TinyLlama-1.1B model to healthcare language through continued pre-training on raw domain text. By exposing the model to medical terminology, clinical language patterns, and healthcare concepts before instruction tuning, we give it a stronger domain foundation. Stage 2 will then build on this adapter to teach the model to follow instruction-style Q&A prompts.

In [3]:
# Install required libraries
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "trl<0.9.0" peft accelerate bitsandbytes

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-tkiugn24/unsloth_cdcf58112cc94c3e9de0f6319b0af223
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-tkiugn24/unsloth_cdcf58112cc94c3e9de0f6319b0af223
  Resolved https://github.com/unslothai/unsloth.git to commit 2246a6c9ae5776092bc77df035b2578cdeea0f62
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached trl-0.8.6-py3-none-any.whl.metadata (11 kB)
Using cached trl-0.8.6-py3-none-any.whl (245 kB)
  Attempting uninstall: trl
    Found existing installation: trl 0.24.0
    Uninstalling trl-0.24.0:
      Successfully uninstalled trl-0.24.0


## 1. Load and Preprocess Raw Domain Text

In [4]:
import re

with open("/content/sample_data/non_instruction_data.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

# Split into paragraphs and filter short ones
paragraphs = [p.strip() for p in raw_text.split("\n\n") if len(p.strip()) > 100]
print(f"Total paragraphs loaded: {len(paragraphs)}")
print("\nSample paragraph:")
print(paragraphs[5])

Total paragraphs loaded: 64

Sample paragraph:
The common cold is caused by viruses, most frequently rhinoviruses. Symptoms include a runny or stuffy nose, sore throat, cough, and mild fever. Most cases resolve within 7 to 10 days without medical treatment. Rest, staying hydrated, and over-the-counter medications for symptom relief can help manage discomfort. Antibiotics are ineffective against viral infections such as the common cold.


In [5]:
from datasets import Dataset

dataset = Dataset.from_dict({"text": paragraphs})
print(dataset)

Dataset({
    features: ['text'],
    num_rows: 64
})


## 2. Load Base Model with Unsloth

In [6]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 512
dtype = None          # Auto-detect
load_in_4bit = True   # QLoRA: 4-bit quantization

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/tinyllama-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)
print("Model loaded successfully.")
print(f"Model parameters: {model.num_parameters():,}")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.6.9: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/762M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.23k [00:00<?, ?B/s]

...

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/438 [00:00<?, ?B/s]

Unsloth: Will load unsloth/tinyllama-bnb-4bit as a legacy tokenizer.


Model loaded successfully.
Model parameters: 1,100,048,384


## 3. Apply LoRA / QLoRA Adapters

In [7]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,                    # LoRA rank
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,           # Scaling factor
    lora_dropout = 0.05,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 42,
    use_rslora = False,
    loftq_config = None,
)
print("LoRA adapters applied.")
model.print_trainable_parameters()

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.6.9 patched 22 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


LoRA adapters applied.
trainable params: 12,615,680 || all params: 1,112,664,064 || trainable%: 1.1338


## 4. Tokenize Dataset

In [8]:
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=max_seq_length,
        padding="max_length",
    )

tokenized_dataset = dataset.map(tokenize_function, batched=True)
tokenized_dataset = tokenized_dataset.remove_columns(["text"])
tokenized_dataset.set_format("torch")
print(f"Tokenized dataset: {tokenized_dataset}")

Map:   0%|          | 0/64 [00:00<?, ? examples/s]

Tokenized dataset: Dataset({
    features: ['input_ids', 'attention_mask'],
    num_rows: 64
})


## 5. Configure and Run Training

In [12]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 42,
        output_dir = "outputs_non_instruction",
    ),
)

print("Starting non-instruction fine-tuning...")
trainer_stats = trainer.train()
print("Training complete!")
print(f"Training loss: {trainer_stats.training_loss:.4f}")

Map (num_proc=2):   0%|          | 0/64 [00:00<?, ? examples/s]

TypeError: Trainer.__init__() got an unexpected keyword argument 'tokenizer'

## 6. Save the Adapter

In [11]:
model.save_pretrained("../models/non_instruction_adapter")
tokenizer.save_pretrained("../models/non_instruction_adapter")
print("Adapter saved to ../models/non_instruction_adapter")

Unsloth: Restored added_tokens_decoder metadata in ../models/non_instruction_adapter/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in ../models/non_instruction_adapter.


Adapter saved to ../models/non_instruction_adapter


## 7. Test the Model After Non-Instruction Fine-Tuning

In [13]:
FastLanguageModel.for_inference(model)

test_prompts = [
    "Hypertension is",
    "Type 2 diabetes is characterized by",
    "Regular exercise helps",
    "Symptoms of a heart attack include",
    "The immune system protects the body by",
]

print("=" * 60)
print("Model outputs after non-instruction fine-tuning:")
print("=" * 60)

for prompt in test_prompts:
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = model.generate(
        **inputs,
        max_new_tokens=80,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
    )
    generated = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    print(f"\nPrompt: {prompt}")
    print(f"Continuation: {generated}")
    print("-" * 40)

Model outputs after non-instruction fine-tuning:


Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/d


Prompt: Hypertension is
Continuation: the tension that is applied to the surface of a tablet, to table, and to a book. (As opposed to the book, the book, and the book.) It is the tension that is applied to the surface of a tablet, to table, and to a book. (As opposed to the book, the book, and the book.) It is the tension that
----------------------------------------


Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Prompt: Type 2 diabetes is characterized by
Continuation: the presence of hypogograms in the form of hypograms.
While it is true that there are more hypograms than hypergrams in the world, the fact that there are more hypograms than hypergrams is not a good thing.
The number of hypergrams is always less than the number of hypograms.
Hypograms are more
----------------------------------------


Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Prompt: Regular exercise helps
Continuation: keep.
There are many reasons why you should consider taking up boxing as a hobby. It's a great workout that can be done in less than an hour. There are a number of benefits that you can get from taking up boxing, such as building your self-confidence, learning self-defense, and even becoming a better athlete.
In the
----------------------------------------


Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Prompt: Symptoms of a heart attack include
Continuation: a combination of cholesteral blood, the cholesteral blood, the cholesteral blood, the cholesteral blood, the cholesteral blood, the cholesteral blood, the cholesteral blood, the cholesteral blood, the cholesteral blood, the cholesteral blood, the cholesteral blood, the
----------------------------------------

Prompt: The immune system protects the body by
Continuation: the defense system. The defense system protects the body by the immune system. The immune system is a complex network of cells, tissues, and organs that work together to protect us from disease.
DESCRIPTION OF THE INVENTION
[0001] This invention relates to the field of immune system research.
[000
----------------------------------------


## Summary

After completing Stage 1 non-instruction fine-tuning, the model should now complete healthcare-domain sentences more fluently and use domain-specific terminology compared to the base model. The LoRA adapter trained here has been saved to `../models/non_instruction_adapter` and captures the healthcare language patterns learned from the raw domain text.

Key observations to check:
- The model should produce medically coherent sentence completions for the test prompts above.
- Outputs should reflect clinical vocabulary and phrasing rather than generic completions.
- Training loss should decrease steadily over the 60 steps, indicating the model is adapting to the domain.

This adapter prepares the model for **Stage 2: Instruction Fine-Tuning**, where it will learn to follow structured Q&A prompts in a healthcare FAQ format.